# Exercise 07: Retrieval-Augmented Generation

> **Chapter:** Ch07 - Retrieval-Augmented Generation
> **Estimated time:** ~55 minutes
>
> This exercise is optional. No submission, no grading.
> The solution notebook is released one week after the exercise.

A generator can only answer from what retrieval puts in front of it. If a fact never reaches the
context, no model and no prompt can recover it. In this exercise you look at the retrieval side of
a RAG system and ask what one retrieval pass can reach. You work with the knowledge base and the
hybrid retrieval chain of the chapter, and with a benchmark of 24 questions that lists the facts
each answer needs. You first measure how much a larger context budget helps. You then build a
retriever with a second hop, the hard-wired loop the chapter calls a synthesizer, and test whether
it is really better. No language model is involved, so everything runs offline and gives the same
numbers on every machine.

**Your tasks** (tick them off as you go):

- [ ] **Quiz** - 20 questions in the quiz app
- [ ] 💻 **C1: Filling a context budget** - assemble the passages a generator receives (in this notebook)
- [ ] ✏ **T1: More context, same gaps** - explain what a larger budget cannot fix
- [ ] 💻 **C2: A second hop** - a two-hop retriever in `tasks.py` that meets a recall target
- [ ] ✏ **T2: Better, but proven?** - read the paired sign test on your own improvement
- [ ] ✏ **T3: A fixed flow for every question** - where the hard-wired hop breaks, and what an agent changes

---

## Quiz

Open the quiz app and work through the **20 questions** for this chapter. On the start screen,
pick the topic **"07 - Retrieval Augmented Generation"**:

**Quiz app:** https://roger-weber.github.io/mmir-unibasel-hs26/quiz/

The quiz covers definitions and basic concepts. The tasks below go further: you measure which
facts a retrieval pass delivers, build the multi-hop retrieval of the synthesizer yourself, and
test your improvement the way the chapter compares two configurations.

---

## The knowledge base and the benchmark

The knowledge base holds 38 Wikipedia articles around the 2025 Nobel Prizes. Each article is
pinned to a fixed revision, so everyone retrieves the same text. It covers the 2025 and 2026
prizes, the 2025 physics laureates with their research and universities, the other 2025
laureates, the 2024 physics laureates, and general articles about the prize. Each section of an
article is split at two levels, as in the chapter: **parents** of at most 2,000 characters, the
passages a generator receives, and **children** of about 400 characters, which drive the dense
search.

The class `KnowledgeBase` in <a href="../shared/nobel_rag.py">shared/nobel_rag.py</a> builds the
retrieval chain of the chapter's pipeline figure. BM25 searches the parents. A bi-encoder
(all-MiniLM-L6-v2) searches the children, and each child is resolved to its parent. Reciprocal
rank fusion merges both lists, and a cross-encoder reranks the top 30. You use four of its
methods:

- `kb.search(query, k)` returns the ids of the top `k` parents, best first. Every call counts as
  one retrieval pass in `kb.searches`.
- `kb.text(pid)` returns the text of parent `pid`, and `kb.label(pid)` its article and section.
- `kb.persons(pid)` returns the person names that the spaCy entity recognizer finds in parent
  `pid`, in order of appearance. It costs no retrieval pass.

> **First run downloads data.** The articles are fetched from Wikipedia once (about 30 seconds)
> and cached. The two retrieval models are about 180 MB together. If you ran the chapter demos,
> they are already cached. Building the index then takes about a minute on a laptop CPU.

In [ ]:
# Standard setup - run this first
import sys, pathlib, math

# Make `shared/` importable regardless of notebook depth
# (exercise notebooks live in exercises/chNN/, solutions in exercises/chNN/solution/).
_here = pathlib.Path().resolve()
_p = _here
while not (_p / "shared").is_dir() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))       # the shared/ root, for `from shared...`
sys.path.insert(0, str(_here))    # the notebook's own folder wins for `import tasks`

%load_ext autoreload
%autoreload 2

import numpy as np

from shared.display import display_md, print_table
from shared.nobel_rag import KnowledgeBase, BENCHMARK, has, context_recall

kb = KnowledgeBase()
LENGTHS = [len(kb.text(pid)) for pid in range(len(kb.parents))]
display_md(f"**{len(kb.collection)} sections** split into **{len(kb.parents)} parents** "
           f"(mean {np.mean(LENGTHS):,.0f} characters) and **{len(kb.children)} children**.")

The benchmark is the one of the chapter's evaluation demo. Each question comes with a list of
**expected facts**: short strings such as a name or a university that a complete answer must
contain. Twelve questions need one lookup. The other twelve are multi-hop: the answer needs a
fact about a person whom the question does not name.

A fact counts as present if it appears in the text as whole words, ignoring case and accents.
**Context recall** is the share of a question's expected facts that appear somewhere in the
context, as in the chapter's section on evaluating RAG.

In [ ]:
print_table([[qid, kind, q, ", ".join(facts)] for qid, kind, q, facts in BENCHMARK],
            headers=["Id", "Kind", "Question", "Expected facts"])

## Filling a context budget

The generator does not receive the whole ranking. It receives as many passages as fit into its
context budget. Here we measure that budget in characters. Real systems count tokens, but
characters keep the arithmetic visible.

> **How this works:** Implement the function in the stub cell (replace `# YOUR CODE HERE`). Run
> the verify cell; the assertions either pass or fail with a specific message. Write the code
> yourself or direct an AI to implement it; the goal is to understand the result.

### 💻 Task C1 - Filling a context budget

Write `fill_context(ranked, lengths, budget)`. It walks through the ranked passage ids in order
and returns the ids that go into the context:

- `lengths[pid]` is the length of passage `pid` in characters.
- A passage that appears again later in `ranked` is taken only once.
- A passage that no longer fits into the remaining budget is skipped, and the walk goes on, since
  a later, shorter passage may still fit.
- The result keeps the ranked order, and its total length never exceeds `budget`.

In [ ]:
def fill_context(ranked: list[int], lengths, budget: int) -> list[int]:
    """The ids of the passages that go into a context of at most `budget` characters."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- Verify C1 ---
sizes = {1: 500, 2: 1500, 3: 300, 4: 800, 5: 2500}
got = fill_context([1, 2, 3], sizes, 10_000)
assert got is not None, "fill_context returns None, did you forget `return`?"
assert got == [1, 2, 3], f"everything fits: expected [1, 2, 3], got {got}"
assert fill_context([], sizes, 1000) == [], "an empty ranking gives an empty context"
assert fill_context([1, 2], sizes, 0) == [], "a budget of 0 admits nothing"
assert fill_context([1, 1, 3, 1], sizes, 10_000) == [1, 3], "take each passage only once"
got = fill_context([5, 1, 2, 3], sizes, 2000)
assert got == [1, 2], f"passage 5 is too long, skip it and go on: expected [1, 2], got {got}"
got = fill_context([1, 2, 4, 3], sizes, 2300)
assert got == [1, 2, 3], f"4 no longer fits but 3 does: expected [1, 2, 3], got {got}"
assert fill_context([1, 2], sizes, 2000) == [1, 2], "a context of exactly the budget is allowed"
for qid, _, q, _ in BENCHMARK[:6]:
    for budget in (2000, 8000):
        ids = fill_context(kb.search(q, 60), LENGTHS, budget)
        assert sum(LENGTHS[i] for i in ids) <= budget, f"{qid}: the context exceeds {budget}"
        assert len(set(ids)) == len(ids), f"{qid}: a passage appears twice"
print("✓ fill_context correct!")

The explore cell runs a single retrieval pass for every question and fills contexts from 2,000 up
to 32,000 characters. That is 1 to 16 full parents. It reports the mean context recall for each
kind of question.

In [ ]:
# --- Explore C1 ---
BUDGETS = [2000, 4000, 8000, 16000, 32000]
RANKED = {qid: kb.search(q, 60) for qid, _, q, _ in BENCHMARK}   # one retrieval pass each

def single_pass(qid, budget):
    """Context of the single-pass system: the top of one ranking, filled to the budget."""
    return fill_context(RANKED[qid], LENGTHS, budget)

rows = []
for kind in ("single", "multi"):
    row = [kind]
    for budget in BUDGETS:
        rec = [context_recall([kb.text(i) for i in single_pass(qid, budget)], facts)
               for qid, k, _, facts in BENCHMARK if k == kind]
        row.append(f"{np.mean(rec):.2f}")
    rows.append(row)
print_table(rows, headers=["Questions"] + [f"{b:,} chars" for b in BUDGETS])
# How much does each doubling of the budget buy, for each kind of question?
# One single-lookup question loses its fact when the budget grows from 2,000 to 4,000. Which one, and how is that possible?

## More context, same gaps

The cell below takes the multi-hop questions at a budget of 8,000 characters. For each expected
fact that is missing from the context, it finds the first passage in the single-pass ranking that
contains the fact. It reports that passage's rank and name. The cell uses your `fill_context`
from C1, so finish C1 first.

In [ ]:
rows = []
for qid, kind, q, facts in BENCHMARK:
    if kind != "multi":
        continue
    context = [kb.text(i) for i in single_pass(qid, 8000)]
    for fact in facts:
        if any(has(t, fact) for t in context):
            continue
        rank = next((n for n, pid in enumerate(RANKED[qid], 1) if has(kb.text(pid), fact)), None)
        rows.append([qid, q, fact, rank or f"not in the top {len(RANKED[qid])}",
                     kb.label(RANKED[qid][rank - 1]) if rank else "-"])
print_table(rows, headers=["Id", "Question", "Missing fact", "First rank holding it", "That passage"])

The passage that answers m05 is the start of Francis Halzen's article. The next cell prints it in
full, and then looks inside the retrieval chain with `kb.engine_ranks`. It shows where the passage
ranks in BM25 and where each of its children ranks in the dense search, among all 2,682 children.
Remember that the cross-encoder only rereads the top 30 candidates after fusion.

In [ ]:
q = next(q for qid, _, q, _ in BENCHMARK if qid == "m05")
pid = next(p for p in range(len(kb.parents)) if kb.label(p) == "Francis Halzen > Introduction")
ranks = kb.engine_ranks(q, pid)
display_md(f"**{kb.label(pid)}** for m05, *{q}*\n\n> {kb.text(pid)}\n\n"
           f"BM25 rank of the passage: **{ranks['bm25']}**. Final rank after fusion and "
           f"reranking: **{ranks['final']}**.")
print_table([[n, rank, " ".join(text.split())] for n, (rank, text) in enumerate(ranks["children"], 1)],
            headers=["Child", "Dense rank", "Text"])

> **How this works:** Write your answer in the markdown cell below the question (replace
> *Your answer here...*). The solution notebook fills the same cell with a model answer, so you
> can compare and reflect.

### ✏ Task T1 - More context, same gaps

Raising the budget from 8,000 to 32,000 characters completes every single lookup, but multi-hop
recall rises only from 0.71 to 0.81. Use the two cells above to explain why the missing passages
rank so low, although they hold exactly the fact the question asks for. Base your answer on what
the question and the passage have in common, and on what the retrieval models can see.

Then judge a common argument: "Context windows now hold a million tokens, so we can simply pass
more passages and stop worrying about retrieval." What does your table say about it?

> *Your answer here...*

---

## A second hop

The chapter describes the synthesizer as a loop: retrieve, read, decide what is still missing,
form a follow-up query, retrieve again. Its multi-hop example first retrieves the laureates and
then looks up each one. An agent runs such a loop with a language model that writes the follow-up
queries. Here you build the hard-wired version with no model at all. The follow-up queries are the
names of people found in the first hop, so the names act as **bridges** between the question and
the second fact.

> **How this works:** Implement `two_hop` in <a href="tasks.py">tasks.py</a>. Its docstring is the
> spec and lists what the verify cell checks. Write the code yourself or direct an AI from the
> docstring, then run the verify cell. Autoreload picks up each save, so you never restart the
> kernel.

### 💻 Task C2 - A two-hop retriever

Write `two_hop(question, kb, budget, fill)`. It runs a first retrieval pass for the question,
takes person names from what it found as bridges, retrieves again for the bridges, and assembles
one context within the budget. `fill(ranked, budget)` is your `fill_context` from C1, already
bound to the passage lengths. Which passages you take names from, how many bridges you follow,
what you search for, and in which order the passages enter the context are your design decisions.

The verify cell runs your retriever on all 24 questions at a budget of 8,000 characters. It
requires a mean context recall of at least 0.85 on the multi-hop questions, which the single pass
reaches only well beyond 32,000 characters. On the single lookups it must not fall below the
single pass at the same budget. Each question may use at most 6 retrieval passes. Run the verify
cell, adjust your design until it passes, then read the explore cell.

In [ ]:
# --- Verify C2 ---
from tasks import two_hop

BUDGET = 8000
fill = lambda ranked, budget: fill_context(ranked, LENGTHS, budget)

out = two_hop(BENCHMARK[0][2], kb, BUDGET, fill)
assert out is not None, "two_hop returns None, did you forget `return`?"
assert set(out) >= {"context", "bridges"}, f"return a dict with 'context' and 'bridges', got {out!r}"

HOP, BASE = {}, {}
for qid, kind, q, facts in BENCHMARK:
    kb.searches = 0
    out = two_hop(q, kb, BUDGET, fill)
    ids = list(out["context"])
    assert len(set(ids)) == len(ids), f"{qid}: a passage appears twice in the context"
    assert sum(LENGTHS[i] for i in ids) <= BUDGET, f"{qid}: the context exceeds {BUDGET:,} characters"
    assert kb.searches <= 6, f"{qid}: {kb.searches} retrieval passes, at most 6 allowed"
    HOP[qid] = dict(out, context=ids, searches=kb.searches,
                    recall=context_recall([kb.text(i) for i in ids], facts))
    BASE[qid] = context_recall([kb.text(i) for i in single_pass(qid, BUDGET)], facts)

problems, rows = [], []
for kind in ("single", "multi"):
    base = np.mean([BASE[qid] for qid, k, *_ in BENCHMARK if k == kind])
    hop = np.mean([HOP[qid]["recall"] for qid, k, *_ in BENCHMARK if k == kind])
    rows.append([kind, f"{base:.2f}", f"{hop:.2f}"])
    if kind == "multi" and hop < 0.85:
        problems.append(f"multi-hop recall {hop:.2f}, needs at least 0.85")
    if kind == "single" and hop < base - 1e-9:
        problems.append(f"single-lookup recall {hop:.2f}, below the single pass ({base:.2f})")
print_table(rows, headers=["Questions", "Single pass", "Two hops"])
assert not problems, "two_hop misses the target:\n  " + "\n  ".join(problems)
print("✓ two_hop meets the recall target within the budget!")

In [ ]:
# --- Explore C2 ---
rows = [[qid, kind, f"{BASE[qid]:.2f}", f"{HOP[qid]['recall']:.2f}", HOP[qid]["searches"],
         ", ".join(HOP[qid]["bridges"]) or "-"] for qid, kind, *_ in BENCHMARK]
print_table(rows, headers=["Id", "Kind", "Single pass", "Two hops", "Searches", "Bridges followed"])
# Which questions gain, which lose? Which bridges are not people at all, or not the right people?

## Better, but proven?

Your retriever raised the mean multi-hop recall clearly. The chapter warns that a higher average
is not yet proof. The paired sign test compares two systems only on the questions where they
disagree. Here a question counts as a win for a system if its context is complete, with every
expected fact present, and the other system's context is not.

In [ ]:
def sign_test(wins_b, wins_a):
    """Two-sided paired sign test: p-value of a split at least this uneven."""
    n, k = wins_a + wins_b, max(wins_a, wins_b)
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k, n + 1)) / 2 ** n)

complete_base = {qid: BASE[qid] == 1.0 for qid, *_ in BENCHMARK}
complete_hop = {qid: HOP[qid]["recall"] == 1.0 for qid, *_ in BENCHMARK}
hop_wins = [q for q in complete_hop if complete_hop[q] and not complete_base[q]]
base_wins = [q for q in complete_hop if complete_base[q] and not complete_hop[q]]
both = sum(complete_hop[q] and complete_base[q] for q in complete_hop)
print_table([[sum(complete_base.values()), sum(complete_hop.values()), both,
              f"{len(hop_wins)} ({', '.join(hop_wins) or '-'})",
              f"{len(base_wins)} ({', '.join(base_wins) or '-'})",
              f"{sign_test(len(hop_wins), len(base_wins)):.3f}"]],
            headers=["Complete: single pass", "Complete: two hops", "Complete: both",
                     "Only two hops", "Only single pass", "p"])

### ✏ Task T2 - Better, but proven?

Your two-hop retriever reaches a much higher multi-hop recall than the single pass at the same
budget. Yet the sign test above does not show a significant difference at the usual threshold of
0.05. Explain how both can be true. Use the numbers of the table: how many questions decide the
test, and what happens to the others?

Then answer two questions. How many disagreements would the test need before even a unanimous
split could reach $p < 0.05$, and roughly how large a benchmark would that take at the
disagreement rate you observed? And consider how you arrived at your design in C2: why should a
result on these 24 questions be read with extra care?

> *Your answer here...*

---

## A fixed flow for every question

The two-hop retriever runs the same chain for every question: one search, names from the top
passage, one search per name. In the chapter's terms this is a pipeline hard-wired for one
intent, here run on every question, and it is what the agent loop replaces.

### ✏ Task T3 - A fixed flow for every question

Look at the bridges your retriever followed in the explore cell of C2. Pick two questions where
the hard-wired hop fails, or helps only by luck, and explain what goes wrong in each. Then describe
what the agent loop of the chapter would do differently on these two questions, and what that
costs compared with your retriever.

> *Your answer here...*